Now for a complex fit - many parameters, one outcome ...

In [1]:
import csv
import numpy as np
import math as m
from scipy import optimize

Probably the major difference - the inputs are now stored in a vector, and so are the weights and the bias ...

In [2]:
def costFun(params):
    cfsum = 0.0
    vecW=np.zeros(8,'f')
    vecW=[params[0],params[1],params[2],params[3],params[4],params[5],params[6],params[7]]
    for lidx in range(len(xindep)):
        vecX=xindep[lidx,:]
        expArg=np.dot(vecW,vecX)
        LTerm=-ydep[lidx]*m.log(1/(1+m.exp(expArg+params[8])))\
                  -(1-ydep[lidx])*m.log(1-1/(1+m.exp(expArg+params[8])));
        cfsum += LTerm
    cfsum /= lidx    
    return cfsum

In [3]:
with open('diabetesTraining.csv') as f:
    linecount = sum(1 for line in f)
    
with open('diabetesTraining.csv',newline='\n') as f:
    reader = csv.reader(f)
    diabetesdata=np.zeros([linecount-1,9],'f')
    counter = 0
    for row in reader:
        if counter != 0:
            diabetesdata[counter-1,0]=row[0]
            diabetesdata[counter-1,1]=row[1]
            diabetesdata[counter-1,2]=row[2]
            diabetesdata[counter-1,3]=row[3]
            diabetesdata[counter-1,4]=row[4]
            diabetesdata[counter-1,5]=row[5]
            diabetesdata[counter-1,6]=row[6]
            diabetesdata[counter-1,7]=row[7]
            diabetesdata[counter-1,8]=row[8]
        counter+=1

In [4]:
# Save the original min and max of each training feature before normalization.
trainingMin = np.min(diabetesdata[:, :8], axis=0) # Minimum of each of the 8 columns
trainingMax = np.max(diabetesdata[:, :8], axis=0) # Maximum of each of the 8 columns

xindep=diabetesdata[0:,:8]
for colidx in range(8):
    mincol=min(xindep[:,colidx])
    for rowidx in range(linecount-1):
        xindep[rowidx,colidx]-=mincol
    maxcol=max(xindep[:,colidx])
    for rowidx in range(linecount-1):
        xindep[rowidx,colidx]/=maxcol
ydep=diabetesdata[0:,8]

In [5]:
firstGuess = [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0]
result = optimize.minimize(costFun, firstGuess, method='nelder-mead')
print("Weights")
print("# of pregnancies: ", result.x[0])
print("Blood glucose: ", result.x[1])
print("Blood pressure: ", result.x[2])
print("Skin thickness: ", result.x[3])
print("Insulin in serum: ", result.x[4])
print("BMI: ", result.x[5])
print("Genetic risk score: ", result.x[6])
print("Age: ", result.x[7])
print("Bias: ",result.x[8])

Weights
# of pregnancies:  0.43110764263268414
Blood glucose:  -6.971084093998076
Blood pressure:  -0.7114539320851416
Skin thickness:  -0.16181767197855612
Insulin in serum:  0.7626279451911149
BMI:  0.014363062129206001
Genetic risk score:  0.4237572828438927
Age:  -0.2449276280406296
Bias:  5.298332685495984


In [6]:
with open('diabetesVerification.csv') as f:
    linecountV = sum(1 for line in f)

with open('diabetesVerification.csv', newline='\n') as f:
    reader = csv.reader(f)
    verificationdata = np.zeros([linecountV-1, 9], 'f')
    counter = 0
    for row in reader:
        if counter != 0:
            for colidx in range(9):
                verificationdata[counter-1, colidx] = row[colidx]
        counter += 1

In [7]:
xverify = verificationdata[0:, :8] # Select the 8 input features

# Normalize verification data using the original training-data ranges.
for colidx in range(8):
    mincol = trainingMin[colidx]  # Use the training minimum for this feature
    maxcol = trainingMax[colidx] - mincol # Use the training range for this feature
    for rowidx in range(linecountV-1):
        xverify[rowidx, colidx] -= mincol # Subtract the training minimum
        xverify[rowidx, colidx] /= maxcol # Divide by the training range

yverify = verificationdata[0:, 8] # Actual outcomes for comparison

In [ ]:
vecW = result.x[:8] # Reuse the 8 weights learned during training
bias = result.x[8] # Reuse the learned bias

correct = 0

for lidx in range(len(xverify)):
    vecX = xverify[lidx, :] # Features of one verification patient
    expArg = np.dot(vecW, vecX) # Weighted sum of the input features
    prediction = 1 / (1 + m.exp(expArg + bias)) # Apply the same sigmoid as in training

    if round(prediction) == yverify[lidx]:  # Compare predicted class with actual outcome
        correct += 1

print("Accuracy:", correct / len(yverify)) # Fraction of correct predictions

Accuracy: 0.7102803738317757
